### Ingestión del archivo "language.csv"

In [0]:
%run "../includes/configuration"


In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment","")
v_environment = dbutils.widgets.get("p_environment")


### Paso 1 - Leer el archivo CSV usando "DataFrameReader" de Spark
### La documentación para los comandos de Spark la sacamos de https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html
### Este notebook está basado en el 01-Ingestion File Movie, ya que tenemos que hacer lo mismo para el language.csv, sin embargo aca lo haremos más directo y no voy a comentar, en el notebook 01 hay pasos extras informativos y esta todo comentado

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType








In [0]:
language_schema = StructType(fields = [
    StructField("languageId", IntegerType(), False),
    StructField("languageCode", StringType(), True),
    StructField("languageName", StringType(), True)
])

In [0]:
#language_df = spark.read.option("header",True).schema(language_schema).csv("abfss://bronze@moviee.dfs.core.windows.net/language.csv")

language_df = spark.read.option("header",True).schema(language_schema).csv(f"{bronze_folder_path}/language.csv")




In [0]:
language_df.printSchema()

In [0]:
display(language_df)

### Paso 2 - Seleccionar solo las columnas requeridas

In [0]:
from pyspark.sql.functions import col






In [0]:
languages_selected_df = language_df.select(col("languageId"),col("languageName"))

In [0]:
display(languages_selected_df)

### Paso 3 - Cambiar el nombre de las columnas según lo requerido

In [0]:
languages_renamed_of = languages_selected_df \
                        .withColumnRenamed("languageId", "language_id") \
                        .withColumnRenamed("languageName", "language_name")



                        



In [0]:
display(languages_renamed_of)

### Paso 4 - Agregar la columnas "ingestion_date" y "environment" al DataFrame

In [0]:
from pyspark.sql.functions import current_timestamp, lit











In [0]:
#languages_final_df = languages_renamed_of.withColumn("ingestion_date", current_timestamp()) \
#                                    .withColumn("env", lit("production"))


languages_final_df = add_ingestion_date(languages_renamed_of) \
                    .withColumn("environment", lit(v_environment))








                                    

In [0]:
display(languages_final_df)

### Paso 5 - Escribir datos en el datalake en formato "Parquet"

In [0]:
#languages_final_df.write.mode("overwrite").parquet("abfss://silver@moviee.dfs.core.windows.net/languages")

languages_final_df.write.mode("overwrite").parquet(f"{silver_folder_path}/languages")










In [0]:
#display(spark.read.parquet("abfss://silver@moviee.dfs.core.windows.net/languages"))

display(spark.read.parquet(f"{silver_folder_path}/languages"))

### Paso 6 - Escribir datos en una managed table en el contenedor silver

#### Esto es un cambio posterior en el notebook, pasaría a reemplazar al paso 5, ya que ahora no quiero más generar archivos de salida en la capa silver a partir del dataframe, sino meter esa info en una tabla administrada por spark, pero dejo igual lo del paso 5, ya que esto se crea dentro de la carpeta _unitystorage y no pisa lo del paso 5

In [0]:
languages_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.languages")

In [0]:
%sql
SELECT * FROM movie_silver.languages

In [0]:
dbutils.notebook.exit("Success")